[<img src="../quantumsymmetry_logo.png" alt="QuantumSymmetry" width="450"/>](https://github.com/dariopicozzi/quantumsymmetry)

> **Note:** if you are running this notebook on Google Colab, the next cell will install `quantumsymmetry` and its dependencies (this might take a few minutes):

In [1]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip -q install quantumsymmetry
    !pip -q install pylatexenc

# Real-time evolution

The imaginary-time descent behind the [VQE tutorial](tree_03_vqe.ipynb) slides the state downhill in energy; **real-time** evolution is the *same* geometric flow turned through 90 degrees, so it circulates at constant energy instead of descending. The diagonal metric that made the descent free makes this rotated flow free too (formally, the metric is the real part of a Kahler structure whose imaginary part supplies the rotation -- the Wick rotation $\tau \mapsto -it$). We stay with the lithium hydride (4, 4) active space from the VQE tutorial: we prepare its ground state, give it a small dipole kick, and follow the real-time evolution, comparing the variational trajectory against exact propagation.

## The symmetry-adapted ansatz

We rebuild the same symmetry-adapted encoding as before. Real-time evolution needs the *phases* of the state, so this time the ansatz is built with `complex=True`, which appends a second, phase ($R_z$) cascade after the amplitude ($R_y$) one and so roughly doubles the CNOT count (the 56 below, against 28 for the real VQE circuit). Everything else -- the 5-qubit Hamiltonian and the symmetry-adapted support -- is identical to the VQE notebook.

In [2]:
import numpy as np
from scipy.linalg import expm
from qiskit_nature.second_q.drivers import PySCFDriver
from quantumsymmetry import (Encoding, MinimalCircuit, evolve_realtime,
                             project_vqd)

enc = Encoding('Li 0 0 0; H 0 0 1.5957', 'sto-3g', CAS=(4, 4),
               output_format='qiskit')
H = enc.hamiltonian
support = enc.symmetry_adapted_support()

Hm = H.to_matrix()
psi_gs = np.linalg.eigh(Hm)[1][:, 0].astype(complex)

mc = MinimalCircuit(enc.encoded_qubits, support=support, complex=True)
print('encoded qubits :', enc.encoded_qubits)
print('active leaves  :', len(support))

encoded qubits : 5
active leaves  : 18


## The dipole kick

We map the $x$-component of the electronic dipole through the *same* encoding with `enc.apply`. Symmetry tapering keeps only the part of the dipole that preserves the sector, and that part maps the symmetry-adapted support to itself: the kicked state stays inside the support with no leakage, so the pruned ansatz can represent it exactly.

In [3]:
problem = PySCFDriver(atom='Li 0 0 0; H 0 0 1.5957', basis='sto-3g').run()
ops = problem.properties.electronic_dipole_moment.second_q_ops()
mu_x = enc.apply(ops['XDipole']).to_matrix()

psi0 = expm(-1j * 0.05 * mu_x) @ psi_gs
psi0 /= np.linalg.norm(psi0)

leakage = np.linalg.norm(np.delete(psi0, support))
print('leakage off support:', f'{leakage:.1e}')

leakage off support: 8.5e-16


`evolve_realtime` integrates the real-time flow with an explicit first-order Euler step around the shift-rule right-hand side — the real-time twin of one natural-gradient / imaginary-time step. We compare its final state to the exact propagator $e^{-iHT}|\psi_0\rangle$ via the infidelity $1 - |\langle\psi_{\rm exact}|\psi\rangle|^2$.

In [4]:
T, n_steps = 0.4, 40
psi_exact = expm(-1j * T * Hm) @ psi0

traj = evolve_realtime(mc, H, psi0, T, n_steps, method='euler')
fidelity = abs(np.vdot(psi_exact, traj.states[-1])) ** 2

print('final-time infidelity:', f'{1 - fidelity:.2e}')
print('CNOTs                :', mc.circuit.count_ops().get('cx', 0))

final-time infidelity: 4.44e-16
CNOTs                : 56


The variational trajectory tracks the exact evolution at numerical precision.

## Comparison with projected evolution (pVQD)

The same ansatz also drives a *projected* time-evolution scheme: `project_vqd` takes one exact propagation step $e^{-iH\,dt}$ and re-projects it onto the manifold by maximising fidelity at each time. On this small, noiseless active space both integrators sit at the infidelity floor; the paper studies the larger systems and longer times where the metric-based (TDVP) integrator avoids the drift that projection schemes accumulate.

In [5]:
times = np.linspace(0, T, n_steps + 1)
pvqd = project_vqd(mc, H, psi0, times)
pvqd_infidelity = 1 - abs(np.vdot(psi_exact, pvqd.states[-1])) ** 2

print('TDVP infidelity :', f'{1 - fidelity:.2e}')
print('pVQD infidelity :', f'{pvqd_infidelity:.2e}')

TDVP infidelity : 4.44e-16
pVQD infidelity : 2.22e-16


### The series

1. <a href="tree_01_welcome.ipynb" />Welcome: the binary-tree ansatz</a>
2. <a href="tree_02_ansatz_and_metric.ipynb" />The ansatz, its pruning and its diagonal metric</a>
3. <a href="tree_03_vqe.ipynb" />Natural-gradient VQE for molecules</a>
4. <a href="tree_04_dynamics.ipynb" />Real-time evolution</a>
5. <a href="tree_05_sampling.ipynb" />Sector-Haar sampling</a>
6. <a href="tree_06_dressing.ipynb" />Dressing the ansatz: a symmetry-adapted Schrieffer-Wolff transformation</a>
7. <a href="tree_07_spin.ipynb" />Spin adaptation: exact total spin on the tree</a>

<p style="text-align: left"> <a href="tree_03_vqe.ipynb" />&lt; Previous: Natural-gradient VQE for molecules</a> </p>
<p style="text-align: right"> <a href="tree_05_sampling.ipynb" />Next: Sector-Haar sampling &gt;</a> </p>